<a href="https://colab.research.google.com/github/XmanujaX/LLM_Lab04/blob/main/notebooks/LLM_Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1 Install the required libraries
!pip install -q transformers datasets peft trl accelerate bitsandbytes

In [ ]:
# 2 Check GPU is availability
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Available: {gpu_name}")
    print(f"Total GPU Memory: {gpu_memory:.2f} GB")
else:
    print("GPU not found. Please change runtime to T4 GPU.")

GPU Available: Tesla T4
Total GPU Memory: 15.64 GB


In [ ]:
# 3 Choose a Small Instruction-Tuned Model
# Choose Qwen/Qwen2.5-0.5B-Instruct model

In [ ]:
# 4 Create a small dataset (Computer shop)
import pandas as pd

data = [
    {"question": "What is the return policy for laptops?", "answer": "You can return any laptop within 30 days of purchase for a full refund, provided it is in its original packaging."},
    {"question": "Do you offer student discounts?", "answer": "Yes, we offer a 10% discount on all accessories for students with a valid ID."},
    {"question": "How long does a custom PC build take?", "answer": "A custom PC build typically takes 3-5 business days to assemble and test."},
    {"question": "What brands of graphics cards do you sell?", "answer": "We stock NVIDIA and AMD graphics cards from brands like ASUS, MSI, and Gigabyte."},
    {"question": "Is there a warranty on refurbished items?", "answer": "Refurbished items come with a 90-day warranty covering hardware defects."},
    {"question": "Can I trade in my old laptop?", "answer": "Yes, we have a trade-in program. Bring your old laptop for an evaluation and get store credit."},
    {"question": "Do you offer repair services?", "answer": "Yes, we offer in-store repair services for both hardware and software issues."},
    {"question": "What is the price of a basic office PC?", "answer": "Our basic office PC starts at $499 and includes an Intel i3 processor, 8GB RAM, and a 256GB SSD."},
    {"question": "Do you sell monitors?", "answer": "Yes, we sell a range of monitors from 24-inch 1080p to 34-inch ultrawide displays."},
    {"question": "What is the best laptop for gaming?", "answer": "For gaming, we recommend the ASUS ROG Zephyrus with an RTX 4060, priced at $1,299."},
    {"question": "Can I get a discount for buying a full setup?", "answer": "Yes, if you buy a PC, monitor, and peripherals together, we can offer a 5% bundle discount."},
    {"question": "Do you offer data recovery services?", "answer": "We do, but data recovery is a specialized service and prices start at $150."},
    {"question": "What is your opening hours?", "answer": "We are open Monday to Saturday from 9 AM to 7 PM, and Sunday from 11 AM to 5 PM."},
    {"question": "Do you have a website?", "answer": "Yes, you can browse and purchase products on our website, www.example-tech-shop.com."},
    {"question": "Can I pay in installments?", "answer": "Yes, we offer financing options through Affirm for purchases over $500."},
    {"question": "Do you sell keyboards and mice?", "answer": "Yes, we carry a wide selection of mechanical keyboards, gaming mice, and office peripherals."},
    {"question": "What is your phone number?", "answer": "You can reach us at (555) 123-4567."},
    {"question": "Do you have an IT support service for small businesses?", "answer": "Yes, we offer on-site and remote IT support packages for small businesses."},
    {"question": "What is the fastest processor you sell?", "answer": "Our fastest consumer processor is the Intel Core i9-14900K."},
    {"question": "Can I return a custom-built PC?", "answer": "Custom-built PCs are subject to a 15% restocking fee if returned within 14 days."},
    {"question": "Do you offer virus removal?", "answer": "Yes, our virus removal service starts at $79 and includes a full system scan and cleanup."},
    {"question": "What is the cheapest laptop you have?", "answer": "The cheapest laptop we have is a 14-inch Chromebook for $199."},
    {"question": "Do you sell printers?", "answer": "Yes, we sell HP, Canon, and Brother printers, along with ink and toner cartridges."},
    {"question": "Can you build a PC for video editing?", "answer": "Absolutely. We can build a PC optimized for video editing with a fast CPU, ample RAM, and a dedicated GPU."},
    {"question": "What is your price match policy?", "answer": "We will match the price of any major competitor on identical, in-stock items."},
    {"question": "Do you offer a warranty on custom-built PCs?", "answer": "Yes, all our custom-built PCs come with a standard 1-year parts and labor warranty."},
    {"question": "What is the difference between an SSD and an HDD?", "answer": "An SSD (Solid State Drive) is faster and more reliable, while an HDD (Hard Disk Drive) offers more storage for a lower price."},
    {"question": "Can I upgrade the RAM on my laptop?", "answer": "Many laptops have upgradeable RAM, but some are soldered. Check your model's specifications or ask our technicians."},
    {"question": "Do you sell gaming chairs?", "answer": "Yes, we carry a selection of ergonomic gaming chairs from brands like Secretlab and DXRacer."},
    {"question": "What is your shipping policy?", "answer": "We offer free shipping on orders over $50 and standard shipping is $5.99."}
]

# Convert to a Hugging Face Dataset
from datasets import Dataset
dataset = Dataset.from_pandas(pd.DataFrame(data))
print(dataset)

Dataset({
    features: ['question', 'answer'],
    num_rows: 30
})


In [ ]:
# 5 Configer LoRA
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 1. Quantization Configuration (4-bit QLoRA) - FIXED FOR T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # <-- CHANGE: Use float16, not bfloat16
    bnb_4bit_use_double_quant=True,
)

# 2. Load Model and Tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,              # <-- ADD: Ensure model weights are also float16
)
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 3. LoRA Configuration (remains the same)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print("Model loaded with float16 compute dtype and LoRA configured.")

print(" Model loaded in 4-bit and LoRA configured.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded with float16 compute dtype and LoRA configured.
 Model loaded in 4-bit and LoRA configured.


In [ ]:
# 6 Fine-Tune with TRL's SFTTrainer
from trl import SFTTrainer, SFTConfig

# Format the dataset into a single text field
def format_data(example):
    return {
        "text": f"<|im_start|>user\n{example['question']}<|im_end|>\n<|im_start|>assistant\n{example['answer']}<|im_end|>"
    }

formatted_dataset = dataset.map(format_data)

# Training Arguments
training_args = SFTConfig(
    output_dir="./qwen-lab4-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100,
    save_steps=50,
    optim="paged_adamw_8bit",
    bf16=False,
    fp16=True,
    max_length=512,
    gradient_checkpointing=True,
    loss_type="nll",
)

# Initialize Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    args=training_args,
    processing_class=tokenizer,
)

# --- FIX FOR T4 GPU BFLOAT16 ERROR ---
# PEFT creates LoRA adapters in bf16, but the fp16 GradScaler needs fp32 gradients on T4.
# We manually cast the trainable parameters to fp32.
for param in trainer.model.parameters():
    if param.requires_grad and param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)

print("Casted trainable LoRA parameters to float32.")
# -------------------------------------

# Start Training
print("Starting training...")
trainer.train()
print("Training complete.")
# Start Training
print("Starting training...")
trainer.train()
print("Training complete.")

# Save the fine-tuned adapter
trainer.model.save_pretrained("qwen-lab4-finetuned")
tokenizer.save_pretrained("qwen-lab4-finetuned")

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

...

Step,Training Loss
10,2.754619
20,0.662495
30,0.224994
40,0.183139
50,0.116258
60,0.109318
70,0.103595
80,0.100932
90,0.097024
100,0.097119


Training complete.
Starting training...


Step,Training Loss
10,0.106296
20,0.109719
30,0.105908
40,0.104175
50,0.099901
60,0.100832
70,0.097095
80,0.098028
90,0.095148
100,0.094800


Training complete.


('qwen-lab4-finetuned/tokenizer_config.json',
 'qwen-lab4-finetuned/chat_template.jinja',
 'qwen-lab4-finetuned/tokenizer.json')

In [ ]:
# 7 Test the Original vs. Fine-Tuned Model

# 1. Define some test questions
test_questions = [
    "Do you offer student discounts?",
    "What is the return policy for laptops?",
    "How long does a custom PC build take?"
]

# 2. Define the generation function
def get_answer(model, question, max_new_tokens=100):
    prompt = f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant\n")[-1].strip()

# 3. Generate Base Model answers FIRST (while it's still the original model)
print("Generating Base Model answers...")
base_answers = [get_answer(model, q) for q in test_questions]

# 4. NOW load the fine-tuned adapter into the model
from peft import PeftModel
finetuned_model = PeftModel.from_pretrained(model, "qwen-lab4-finetuned")
finetuned_model.eval()

# 5. Generate Fine-Tuned Model answers
print("Generating Fine-Tuned Model answers...")
finetuned_answers = [get_answer(finetuned_model, q) for q in test_questions]

# 6. Print the comparison
for q, base_ans, ft_ans in zip(test_questions, base_answers, finetuned_answers):
    print("="*80)
    print(f"QUESTION: {q}")
    print("-"*40)
    print(f"BASE MODEL:\n{base_ans}")
    print("-"*40)
    print(f"FINE-TUNED MODEL:\n{ft_ans}")
    print("="*80)

Generating Base Model answers...


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Generating Fine-Tuned Model answers...
QUESTION: Do you offer student discounts?
----------------------------------------
BASE MODEL:
Yesuseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruser
----------------------------------------
FINE-TUNED MODEL:
Yes, we offer a 10% discount on all accessories for students with a valid ID.
QUESTION: What is the return policy for laptops?
----------------------------------------
BASE MODEL:
Youuseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruseruser